# EG-SEG-002 — five-model Cityscapes smoke
Thin execution wrapper. It selects the OpenMIM-free compatibility path, rebuilds the approved split policy without re-extraction, stages data to `/content`, and runs 100 optimizer steps per model. This is training-path compatibility evidence, not architecture ranking.

In [ ]:
import json
import re
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

REPOSITORY = "https://github.com/emrealmaoglu/edgeguard-road.git"
EDGEGUARD_EXPECTED_COMMIT = "REPLACE_WITH_REVIEWED_EG_SEG_002_COMMIT_SHA"
PROJECT_ROOT = Path("/content/edgeguard-road")
DRIVE_ROOT = Path("/content/drive/MyDrive/EdgeGuard")
if not re.fullmatch(r"[0-9a-f]{40}", EDGEGUARD_EXPECTED_COMMIT):
    raise ValueError("Enter the exact reviewed EG-SEG-002 commit SHA")
if PROJECT_ROOT.exists():
    raise FileExistsError("Refusing an existing project checkout")
subprocess.run(
    ["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY, str(PROJECT_ROOT)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "checkout", "--detach", EDGEGUARD_EXPECTED_COMMIT], check=True
)
actual = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
dirty = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain=v1"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if actual != EDGEGUARD_EXPECTED_COMMIT or dirty:
    raise RuntimeError("Reviewed checkout identity failed")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)], check=True)
drive.mount("/content/drive")

In [ ]:
COMPAT = Path("/content/edgeguard-compatibility")
COMPAT_LOGS = Path("/content/edgeguard-logs")
compatibility_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/train/install_semantic_stack.py"),
    "--config",
    str(PROJECT_ROOT / "configs/training/segmentation/framework_mmseg.yaml"),
    "--project-root",
    str(PROJECT_ROOT),
    "--project-commit",
    EDGEGUARD_EXPECTED_COMMIT,
    "--config-root",
    str(PROJECT_ROOT / "configs/training/segmentation"),
    "--checkout-root",
    "/content/edgeguard-mmseg",
    "--evidence-root",
    str(COMPAT),
    "--log-root",
    "/content/edgeguard-logs",
    "--execute",
]
try:
    subprocess.run(compatibility_command, check=True)
except subprocess.CalledProcessError:
    import torch

    print("=== hosted runtime ===")
    print(
        json.dumps(
            {
                "python": sys.version,
                "torch": torch.__version__,
                "cuda": torch.version.cuda,
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                "disk": shutil.disk_usage("/content")._asdict(),
            },
            indent=2,
            default=str,
        )
    )
    for diagnostic in (COMPAT / "compatibility_failures.json", COMPAT / "run_status.json"):
        if diagnostic.is_file():
            print(f"=== {diagnostic.name} ===")
            print(diagnostic.read_text(encoding="utf-8"))
    for log in sorted(COMPAT_LOGS.rglob("*.stderr.log")):
        lines = log.read_text(encoding="utf-8", errors="replace").splitlines()
        if lines:
            print(f"=== stderr tail: {log.name} ===")
            print("\n".join(lines[-120:]))
    for log in sorted(COMPAT_LOGS.rglob("*.stdout.log")):
        lines = log.read_text(encoding="utf-8", errors="replace").splitlines()
        tail = lines[-120:]
        if any(
            any(marker in line.lower() for marker in ("traceback", "error", "exception", "failed"))
            for line in tail
        ):
            print(f"=== error-bearing stdout tail: {log.name} ===")
            print("\n".join(tail))
    raise
receipt = json.loads((COMPAT / "compatibility_receipt.json").read_text())
INTERPRETER = Path(receipt["interpreter"])
MMSEG = Path(
    "/content/edgeguard-mmseg/mmseg-path-a"
    if receipt["selected_path"] == "hosted_current"
    else "/content/edgeguard-mmseg/mmseg-path-b"
)

In [ ]:
MANIFESTS = DRIVE_ROOT / "manifests/cityscapes/fine/v1"
POLICY = MANIFESTS / "split-policy-v1"
if not POLICY.exists():
    subprocess.run(
        [
            str(INTERPRETER),
            str(PROJECT_ROOT / "scripts/rebuild_cityscapes_splits.py"),
            "--dataset-manifest",
            str(MANIFESTS / "dataset_manifest.json"),
            "--group-summary",
            str(MANIFESTS / "group_summary.json"),
            "--output-directory",
            str(POLICY),
        ],
        check=True,
    )
subprocess.run(
    [
        str(INTERPRETER),
        str(PROJECT_ROOT / "scripts/stage_cityscapes_training.py"),
        "--dataset-root",
        str(DRIVE_ROOT / "datasets/cityscapes/fine/v1"),
        "--dataset-manifest",
        str(MANIFESTS / "dataset_manifest.json"),
        "--split-policy-manifest",
        str(POLICY / "policy_selected_split.json"),
        "--drive-bundle-directory",
        str(DRIVE_ROOT / "datasets/cityscapes/fine/bundles"),
        "--cache-directory",
        "/content/edgeguard-data-cache",
        "--staged-dataset-root",
        "/content/edgeguard-cityscapes-fine",
    ],
    check=True,
)

In [ ]:
subprocess.run(
    [
        str(INTERPRETER),
        str(PROJECT_ROOT / "scripts/train/run_semantic_smoke.py"),
        "--interpreter",
        str(INTERPRETER),
        "--project-root",
        str(PROJECT_ROOT),
        "--project-commit",
        EDGEGUARD_EXPECTED_COMMIT,
        "--config-root",
        str(PROJECT_ROOT / "configs/training/segmentation"),
        "--mmseg-checkout",
        str(MMSEG),
        "--dataset-root",
        "/content/edgeguard-cityscapes-fine",
        "--dataset-manifest",
        str(MANIFESTS / "dataset_manifest.json"),
        "--split-policy-manifest",
        str(POLICY / "policy_selected_split.json"),
        "--run-root",
        "/content/edgeguard-runs/EG-SEG-002",
        "--drive-root",
        str(DRIVE_ROOT),
    ],
    check=True,
)
summary = json.loads(Path("/content/edgeguard-runs/EG-SEG-002/smoke_summary.json").read_text())
if summary["status"] != "ready_for_common_screening":
    raise RuntimeError("EG-SEG-002 smoke promotion gate did not pass")
summary

## Stop
Do not start the common screening campaign in this notebook. SMIYC and full Fishyscapes Lost & Found remain inaccessible here.